# 01 — First Fine-Tune: Qwen 2.5 0.5B on Alpaca (English)

**Purpose:** Run the full QLoRA fine-tuning loop end-to-end on a tiny model with a tiny dataset. This is NOT your real training run — it's a dry run to prove every piece of the pipeline works before you touch Urdu data or the 7B model.

**What you'll learn:**
1. How model loading works (quantization, VRAM)
2. How LoRA adapters attach to a model (which layers, why)
3. How data must be formatted for instruction-tuning (chat templates)
4. How SFTTrainer orchestrates the training loop
5. How to save, reload, and run inference with your fine-tuned model

**Cost:** ~$0.50 on Modal (A100, ~15 min total)

**Runtime:** This notebook is designed for an NVIDIA A100 80GB. It will also work on A100 40GB or even T4/L4 since the 0.5B model is small.

## Step 0: Install Dependencies

Unsloth needs to be installed from source for the latest model support. It patches the model's attention and MLP layers at load time to make training faster — you don't need to change your code, it happens automatically.

**What's being installed:**
- `unsloth`: The speed optimization layer. Sits on top of HuggingFace.
- `trl`: HuggingFace's "Transformer Reinforcement Learning" library. We use its `SFTTrainer` (Supervised Fine-Tuning Trainer) — despite the RL name, SFT is just supervised learning.
- `xformers`: Memory-efficient attention kernels. Unsloth uses these under the hood.

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

## Step 1: Load the Base Model

This is where the magic of QLoRA starts. Let me explain what's happening:

### What is quantization?
The original Qwen 2.5 0.5B model stores each weight as a 16-bit float (2 bytes). A 7B parameter model would need ~14GB just to load. **4-bit quantization** compresses each weight to 4 bits (0.5 bytes), cutting memory by 4x. The model gets slightly less precise, but research shows the quality loss is minimal.

### What is NF4?
NF4 (NormalFloat 4-bit) is a quantization format specifically designed for neural network weights. It works better than naive 4-bit rounding because neural network weights follow a normal distribution, and NF4 is optimized for that distribution.

### What does `FastLanguageModel` do?
It's Unsloth's wrapper around HuggingFace's `AutoModelForCausalLM`. When you call `from_pretrained`, it:
1. Downloads the model from HuggingFace Hub
2. Quantizes it to 4-bit NF4
3. Patches the attention layers for 2x faster training
4. Returns the model + tokenizer ready to go

### Why `max_seq_length = 2048`?
This is the maximum number of tokens (roughly word-pieces) in a single training example. Qwen 2.5 supports up to 128k tokens, but longer sequences = more VRAM. 2048 is plenty for instruction-response pairs and matches your roadmap config.

In [ ]:
from unsloth import FastLanguageModel
import torch

# --- Configuration ---
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # Small model for dry run. Change to 7B later.
MAX_SEQ_LENGTH = 2048
DTYPE = None  # None = auto-detect. Will use bfloat16 on A100, float16 on older GPUs.
LOAD_IN_4BIT = True  # QLoRA = 4-bit base weights + 16-bit LoRA adapters

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## Step 2: Attach LoRA Adapters

### What is LoRA?
Instead of updating ALL 500 million parameters during training (expensive, slow), LoRA adds tiny "adapter" matrices to specific layers. Think of it like this:

- **Full fine-tune:** Rewriting the entire textbook (all 500M parameters change)
- **LoRA:** Adding sticky notes to specific pages (only ~1-5M new parameters)

The base model stays frozen (4-bit, read-only). Only the small adapter weights get trained (16-bit, trainable). This is why it's called **QLoRA** — **Q**uantized base + **Lo**w-**R**ank **A**daptation.

### Key parameters explained:

- **`r = 16` (rank):** Size of the adapter matrices. Higher = more capacity but more VRAM. 16 is a good default. Think of it as "how much new knowledge can the adapter hold."

- **`lora_alpha = 32` (scaling):** Controls how much the adapter's output influences the model. Rule of thumb: `alpha = 2 * r`. Too high = adapter overpowers base knowledge. Too low = adapter has no effect.

- **`target_modules`:** Which layers get adapters. We target all the attention layers (q/k/v/o_proj) and the MLP layers (gate/up/down_proj). More modules = more capacity but more VRAM.

- **`use_gradient_checkpointing = "unsloth"`:** Trades compute for memory. Instead of storing all intermediate values during the forward pass, it recomputes them during backprop. Unsloth's version is optimized to be faster than the standard implementation.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,  # Unsloth recommends 0 — optimized for it
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
    random_state=42,
)

# Show how many parameters are trainable vs frozen
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} ({100 * trainable / total:.2f}%)")
print(f"Total parameters: {total:,}")
print(f"Adapter size (approx): {trainable * 2 / 1024**2:.1f} MB")  # 2 bytes per float16 param

## Step 3: Load and Format the Dataset

### Why Alpaca format?
Most instruction-tuning datasets use the "Alpaca" format — three fields:
- `instruction`: What the user asks
- `input`: Optional additional context (empty string if none)
- `output`: The ideal response

This is a simple, proven format. Your Urdu dataset will use the same structure.

### What is a chat template?
LLMs don't see "user" and "assistant" — they see a flat stream of tokens. The **chat template** is how you encode the conversation structure into that stream. Qwen 2.5 uses this format:

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of Pakistan?<|im_end|>
<|im_start|>assistant
The capital of Pakistan is Islamabad.<|im_end|>
```

`<|im_start|>` and `<|im_end|>` are special tokens the model was pretrained to understand. If you use the wrong template, the model gets confused — this is the #1 mistake first-timers make.

### Why only 500 examples?
This is a dry run. We want to see:
1. Does the loss go down? (Training is working)
2. Does inference produce coherent text? (The template is correct)
3. Does the save/load cycle work? (Pipeline is complete)

We don't need good outputs — we need a working pipeline.

In [ ]:
from datasets import load_dataset

# Load Alpaca-cleaned — a well-known instruction dataset (~52k English examples)
# We only take 500 for this dry run
dataset = load_dataset("yahma/alpaca-cleaned", split="train[:500]")

print(f"Dataset size: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"\n--- Example ---")
print(f"Instruction: {dataset[0]['instruction']}")
print(f"Input: {dataset[0]['input']}")
print(f"Output: {dataset[0]['output'][:200]}...")

In [ ]:
def format_alpaca_to_chat(example):
    """
    Convert Alpaca format -> Qwen chat template.
    
    This function takes {"instruction", "input", "output"} and produces
    a single "text" field with the full chat-formatted string that the
    model will train on.
    
    The tokenizer.apply_chat_template() handles the <|im_start|>/<|im_end|>
    tokens automatically — never hardcode these yourself.
    """
    # Build the user message: instruction + optional input
    user_message = example["instruction"]
    if example.get("input", "").strip():
        user_message += f"\n\n{example['input']}"
    
    # Build the conversation in the format the tokenizer expects
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": example["output"]},
    ]
    
    # apply_chat_template converts this to the proper token format
    # tokenize=False means we get a string back, not token IDs
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,  # False because we include the assistant response
    )
    
    return {"text": text}

# Apply formatting to entire dataset
dataset = dataset.map(format_alpaca_to_chat)

# Inspect what the model will actually see during training
print("--- Formatted example (what the model trains on) ---")
print(dataset[0]["text"][:500])
print("...")

## Step 4: Test the Base Model BEFORE Training

This is important. You need to see what the model does **before** you train it, so you can tell if training made things better or worse. This is your "before" photo.

We'll ask it a simple question and save the response for comparison later.

In [ ]:
# Switch model to inference mode (disables training-specific optimizations)
FastLanguageModel.for_inference(model)

test_prompts = [
    "Explain what photosynthesis is in simple terms.",
    "Write a short poem about mountains.",
    "What are three benefits of exercise?",
]

print("=" * 60)
print("BASE MODEL RESPONSES (before training)")
print("=" * 60)

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]
    
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # True here = we want the model to generate
    )
    
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    
    # Decode only the NEW tokens (skip the input prompt)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    print(f"\n>>> {prompt}")
    print(f"{response.strip()}")
    print("-" * 40)

## Step 5: Train!

### What is SFTTrainer?
SFTTrainer (Supervised Fine-Tuning Trainer) is HuggingFace TRL's training loop. It handles:
- Tokenizing your text data
- Batching and padding
- Forward pass, loss computation, backpropagation
- Learning rate scheduling
- Logging and checkpointing

You configure it, call `.train()`, and it handles the rest.

### Key training parameters explained:

- **`per_device_train_batch_size = 2`:** How many examples the GPU processes at once. Limited by VRAM.

- **`gradient_accumulation_steps = 4`:** Simulates a larger batch by accumulating gradients over 4 mini-batches before updating weights. Effective batch size = 2 × 4 = 8. Larger effective batch = more stable training.

- **`learning_rate = 2e-4`:** How big each weight update is. Too high = training diverges (loss explodes). Too low = training is too slow. 2e-4 is the standard starting point for QLoRA.

- **`lr_scheduler_type = "cosine"`:** Learning rate starts at 2e-4, gradually decreases following a cosine curve. This helps the model "settle" into good weights near the end of training.

- **`warmup_ratio = 0.03`:** For the first 3% of training steps, the learning rate ramps up from 0 to 2e-4. This prevents early instability when the adapter weights are random.

- **`max_steps = 100`:** We cap at 100 steps for this dry run. A full run on 50k examples for 2 epochs would be ~6,250 steps.

### What to watch:
- **Loss should go DOWN.** If it goes up or stays flat, something is wrong.
- Starting loss for instruction-tuning is typically 1.5–3.0
- After training, loss should be 0.5–1.5
- If loss drops to near 0: you're overfitting (memorizing, not learning)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="./output",
        
        # --- Batch size ---
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch = 2 * 4 = 8
        
        # --- Learning rate ---
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        
        # --- Duration ---
        max_steps=100,  # Short! Just a dry run. Remove this for full training.
        
        # --- Precision ---
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        
        # --- Logging ---
        logging_steps=5,  # Print loss every 5 steps
        
        # --- Optimizer ---
        optim="adamw_8bit",  # 8-bit Adam saves VRAM vs standard Adam
        
        # --- Misc ---
        weight_decay=0.01,
        max_grad_norm=1.0,
        seed=42,
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",  # Column name from our formatting step
    ),
)

print("Training configuration ready.")
print(f"Effective batch size: {2 * 4} = {2} per_device × {4} grad_accum")
print(f"Total steps: 100")
print(f"Estimated time: ~2-5 minutes on A100")

In [ ]:
# This is where the actual training happens.
# Watch the loss column — it should decrease over time.
trainer_stats = trainer.train()

print(f"\n--- Training Complete ---")
print(f"Total steps: {trainer_stats.global_step}")
print(f"Final training loss: {trainer_stats.training_loss:.4f}")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.0f} seconds")

## Step 6: Test the Fine-Tuned Model

Same prompts as before. Compare the responses. Don't expect miracles from 500 examples and 100 steps — the point is to confirm the pipeline works and the model didn't break.

**What "working" looks like:** Coherent responses in a slightly different style than before.

**Red flags:** Gibberish, repeated tokens, empty responses, or the model ignoring the question entirely. Any of these means something went wrong in the template or training config.

In [ ]:
# Switch back to inference mode after training
FastLanguageModel.for_inference(model)

print("=" * 60)
print("FINE-TUNED MODEL RESPONSES (after training)")
print("=" * 60)

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]
    
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    print(f"\n>>> {prompt}")
    print(f"{response.strip()}")
    print("-" * 40)

## Step 7: Save the LoRA Adapter

### What gets saved?
Only the adapter weights (~1-20 MB), NOT the full model (~1 GB for 0.5B, ~14 GB for 7B). The adapter is a small file that gets loaded on top of the base model at inference time.

This is the beauty of LoRA — you can share a tiny file instead of a multi-gigabyte model.

### Two ways to save:
1. **Adapter only** (`save_pretrained`): Saves just the LoRA weights. To use it, you load the base model + adapter separately. This is what you push to HuggingFace Hub.
2. **Merged model** (`save_pretrained_merged`): Bakes the adapter into the base model weights. Produces a standalone model that doesn't need the adapter file. Used for deployment.

In [ ]:
# Save the LoRA adapter locally
ADAPTER_DIR = "./output/adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to: {ADAPTER_DIR}")

# Check the adapter size
import os
adapter_size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if f.endswith(('.safetensors', '.bin'))
)
print(f"Adapter size: {adapter_size / 1024**2:.1f} MB")

## Step 8: Save Merged Model (Optional)

This bakes the adapter into the base model, producing a standalone model. This is what you'd deploy to HuggingFace Spaces for the Gradio demo.

For the 0.5B dry run this is fast. For the 7B model, merging takes a few minutes and produces a ~14GB file.

In [ ]:
# Save merged model (adapter baked into base weights) in float16
MERGED_DIR = "./output/merged"
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",  # Full precision merged model
)
print(f"Merged model saved to: {MERGED_DIR}")

## Step 9: Push to HuggingFace Hub (Optional)

Uncomment and run these cells when you're ready to push. You'll need a HuggingFace token — get one at https://huggingface.co/settings/tokens (needs "write" permission).

This step proves the upload flow works before you do it with the real 7B model.

In [ ]:
# --- Uncomment to push to HuggingFace Hub ---
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")  # Or set HF_TOKEN environment variable

# HF_USERNAME = "your-username"  # Change this!

# # Push adapter (small, ~10-20 MB)
# model.push_to_hub(f"{HF_USERNAME}/qwen2.5-0.5b-alpaca-lora", tokenizer=tokenizer)

# # Push merged model (larger, ~1 GB for 0.5B)
# model.push_to_hub_merged(
#     f"{HF_USERNAME}/qwen2.5-0.5b-alpaca-merged",
#     tokenizer,
#     save_method="merged_16bit",
# )

print("Skipped HF push (uncomment when ready)")

## Step 10: VRAM Summary

Check how much GPU memory the whole pipeline used. This tells you whether you'll have headroom when you scale to Qwen 2.5 7B (which uses ~16-20GB in QLoRA mode on A100).

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total_memory = round(gpu_stats.total_mem / 1024**3, 2)

print(f"GPU: {gpu_stats.name}")
print(f"Peak VRAM used: {used_memory} GB / {total_memory} GB ({100 * used_memory / total_memory:.1f}%)")
print(f"\nFor reference:")
print(f"  Qwen 2.5 0.5B QLoRA: ~2-4 GB")
print(f"  Qwen 2.5 7B QLoRA:   ~16-20 GB")
print(f"  A100 80GB has plenty of headroom for the 7B model.")

---

## What You Just Did

You ran the **entire fine-tuning pipeline** end-to-end:

| Step | What happened | Why it matters |
|------|--------------|----------------|
| Load model | Downloaded Qwen 2.5 0.5B, quantized to 4-bit | Proves GPU setup works, quantization works |
| Attach LoRA | Added trainable adapter layers | Proves PEFT configuration is correct |
| Format data | Converted Alpaca → Qwen chat template | Proves template formatting works (most common failure point) |
| Train | Ran SFTTrainer for 100 steps | Proves training loop runs, loss decreases |
| Inference | Generated responses from fine-tuned model | Proves the model produces coherent output after training |
| Save adapter | Saved LoRA weights to disk | Proves save/load cycle works |
| Save merged | Baked adapter into base model | Proves merge works (needed for deployment) |

## What's Next

1. **Notebook 02:** Same pipeline but on **Qwen 2.5 7B** with 1,000 examples — proves the 7B model works with your setup
2. **Week 1:** Build the Urdu dataset (translate Alpaca, pull Aya, hand-craft examples)
3. **Week 2:** Full training on 50k Urdu examples using this exact pipeline, scaled up

## Key Takeaways

- If loss went down and inference produced coherent text: **your pipeline works.** Everything from here is data and scale.
- The chat template is critical. If you change base models, the template changes too. Always verify with a print statement.
- The adapter is tiny compared to the full model. This is why QLoRA is practical on a budget.
- When you move to Urdu, the only things that change are: the dataset and the system prompt. The pipeline stays the same.